<a href="https://colab.research.google.com/github/sflores14/inspirastem2026-bioquimica-computacional/blob/main/preparacion/00_preparacion_colab_y_bioquimica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparación | Google Colab y principios de bioquímica

**InspiraSTEM 2026 | Bioquímica Computacional Aplicada**

Este notebook tiene dos objetivos:

1. familiarizarte con Google Colab y la forma en que trabajaremos durante el workshop;
2. repasar conceptos básicos de bioquímica estructural para que todos comencemos con una base común.

No necesitas experiencia previa en programación. Durante el workshop la mayoría de las celdas estarán preparadas y podrás modificar parámetros científicos mediante controles interactivos.

---

### Cómo trabajar con este notebook

- Ejecuta las celdas de arriba hacia abajo.
- Puedes usar el botón de reproducción a la izquierda de cada celda o presionar `Shift + Enter`.
- Algunas celdas tienen controles que puedes modificar antes de ejecutarlas.
- Si reinicias el runtime, las variables y archivos temporales se borran y deberás volver a ejecutar las celdas desde el inicio.

## 1. Tu primera celda

Google Colab ejecuta código en una computadora remota. Vamos a comenzar con algo sencillo.

Modifica la temperatura y ejecuta la celda.

In [ ]:
#@title Ejecuta tu primera celda
temperature = 287 #@param {type:"slider", min:273, max:350, step:1}

print(f"Temperatura seleccionada: {temperature} K")

El control que acabas de utilizar modifica un **parámetro** que el código utiliza para realizar un cálculo.

Durante el workshop trabajaremos de esta misma forma: tú modificarás parámetros científicos y el notebook se encargará de ejecutar el código necesario.

Si tienes experiencia con Python, siempre puedes abrir una celda y revisar cómo funciona.

## 2. Preparar las herramientas

Esta celda instala y carga las herramientas que utilizaremos en este notebook.

No necesitas modificar nada.

In [ ]:
#@title Preparar el entorno

import sys
import subprocess
import importlib.util
from contextlib import redirect_stdout, redirect_stderr
from io import StringIO

def ensure_package(package, import_name=None):
    import_name = import_name or package
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

ensure_package("rdkit", "rdkit")
ensure_package("py3Dmol", "py3Dmol")
ensure_package("pandas", "pandas")
ensure_package("requests", "requests")

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, Crippen, Lipinski, AllChem
import pandas as pd
import py3Dmol
import requests
from IPython.display import display

print("Entorno listo.")

## 3. De átomos a aminoácidos

Las proteínas están formadas por **aminoácidos**.

Todos los aminoácidos comparten una estructura general con:

- un grupo amino;
- un grupo carboxilo;
- un carbono central;
- una cadena lateral variable, llamada **R**.

La cadena lateral es la parte que cambia entre aminoácidos y determina gran parte de su comportamiento químico.

De forma simplificada, podemos pensar en varias familias:

| Tipo de cadena lateral | Ejemplos |
|---|---|
| Hidrofóbica | Ala, Val, Leu, Ile |
| Aromática | Phe, Tyr, Trp |
| Polar | Ser, Thr, Asn, Gln |
| Positivamente cargada | Lys, Arg |
| Negativamente cargada | Asp, Glu |
| Casos especiales | Gly, Pro, Cys, His |

No necesitas memorizar esta tabla. Lo importante es reconocer que **diferentes cadenas laterales tienen diferentes propiedades químicas**.

### Explora algunos aminoácidos

Selecciona un aminoácido y observa cómo cambia su estructura química.

In [ ]:
#@title Explorar un aminoácido

amino_acid = "Lysine" #@param ["Alanine", "Leucine", "Phenylalanine", "Serine", "Aspartate", "Lysine"]

amino_acids = {
    "Alanine": {
        "smiles": "N[C@@H](C)C(=O)O",
        "character": "hidrofóbico",
        "sidechain": "alifática pequeña"
    },
    "Leucine": {
        "smiles": "N[C@@H](CC(C)C)C(=O)O",
        "character": "hidrofóbico",
        "sidechain": "alifática"
    },
    "Phenylalanine": {
        "smiles": "N[C@@H](Cc1ccccc1)C(=O)O",
        "character": "aromático e hidrofóbico",
        "sidechain": "aromática"
    },
    "Serine": {
        "smiles": "N[C@@H](CO)C(=O)O",
        "character": "polar",
        "sidechain": "contiene un grupo hidroxilo"
    },
    "Aspartate": {
        "smiles": "N[C@@H](CC(=O)O)C(=O)O",
        "character": "ácido / negativamente cargado cerca de pH fisiológico",
        "sidechain": "contiene un carboxilato"
    },
    "Lysine": {
        "smiles": "N[C@@H](CCCCN)C(=O)O",
        "character": "básico / positivamente cargado cerca de pH fisiológico",
        "sidechain": "contiene una amina"
    }
}

info = amino_acids[amino_acid]
mol = Chem.MolFromSmiles(info["smiles"])

display(Draw.MolToImage(mol, size=(420, 260)))
print(f"Aminoácido: {amino_acid}")
print(f"Carácter general: {info['character']}")
print(f"Cadena lateral: {info['sidechain']}")

### Pregunta rápida

Imagina una cavidad dentro de una proteína formada principalmente por residuos hidrofóbicos.

**Cuál esperarías que fuera más compatible con ese ambiente: leucina o aspartato?**

Piensa en tu respuesta antes de continuar.

La idea central es que **la química local del sitio importa**. Una molécula no se une únicamente porque "cabe".

## 4. De aminoácidos a proteínas

Los aminoácidos se conectan mediante **enlaces peptídicos** para formar una cadena polipeptídica.

Podemos describir la organización de una proteína en distintos niveles:

**Estructura primaria**  
La secuencia de aminoácidos.

**Estructura secundaria**  
Patrones locales del esqueleto peptídico, principalmente hélices alfa y hojas beta.

**Estructura terciaria**  
La organización tridimensional completa de una cadena polipeptídica.

**Estructura cuaternaria**  
La asociación de varias cadenas polipeptídicas cuando una proteína está formada por más de una subunidad.

## 5. Explora una proteína real

Usaremos ubiquitina como ejemplo para practicar la visualización molecular.

La estructura se descargará directamente del Protein Data Bank usando el identificador **1UBQ**.

In [ ]:
#@title Cargar una estructura tridimensional

pdb_id = "1UBQ"
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

response = requests.get(url, timeout=30)
response.raise_for_status()
pdb_text = response.text

with open(f"{pdb_id}.pdb", "w") as f:
    f.write(pdb_text)

print(f"Estructura {pdb_id} cargada.")

In [ ]:
#@title Explorar una proteína real

representation = "Sticks" #@param ["Cartoon", "Surface", "Cartoon + Surface", "Sticks"]

view = py3Dmol.view(width=760, height=520)
view.addModel(pdb_text, "pdb")

if representation == "Cartoon":
    view.setStyle({"cartoon": {"color": "spectrum"}})
elif representation == "Surface":
    view.setStyle({"cartoon": {"color": "lightgray"}})
    view.addSurface(py3Dmol.VDW, {"opacity": 0.85})
elif representation == "Cartoon + Surface":
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.addSurface(py3Dmol.VDW, {"opacity": 0.35})
elif representation == "Sticks":
    view.setStyle({"stick": {}})

view.zoomTo()
view.show()

Prueba a:

- rotar la proteína;
- acercar y alejar;
- cambiar entre `Cartoon` y `Surface`.

### Pregunta

**Cambió la proteína al cambiar la representación?**

No. Cambió solamente la manera en que representamos las mismas coordenadas atómicas.

- `Cartoon` facilita reconocer el **fold** y la estructura secundaria.
- `Surface` facilita pensar en forma, accesibilidad y cavidades.
- `Sticks` muestra con mayor detalle los átomos y enlaces.

## 6. Estructura secundaria

En una representación tipo `Cartoon`, las hélices alfa y las hojas beta se vuelven mucho más fáciles de reconocer.

Ejecuta la siguiente celda e intenta identificar al menos una región helicoidal y una región beta.

In [ ]:
#@title Visualizar la estructura secundaria

view = py3Dmol.view(width=760, height=520)
view.addModel(pdb_text, "pdb")

view.setStyle({"helix": True}, {"cartoon": {"color": "red"}})
view.setStyle({"sheet": True}, {"cartoon": {"color": "blue"}})
view.setStyle({"cartoon": {"color": "lightgray"}})

# Apply secondary-structure coloring after the base representation
view.setStyle({"ss": "h"}, {"cartoon": {"color": "red"}})
view.setStyle({"ss": "s"}, {"cartoon": {"color": "blue"}})

view.zoomTo()
view.show()

print("Rojo: hélices alfa")
print("Azul: hojas beta")
print("Gris: otras regiones")

## 7. Qué mantiene una proteína plegada?

La estructura tridimensional de una proteína resulta de muchas interacciones actuando al mismo tiempo.

Entre las más importantes se encuentran:

- efecto hidrofóbico;
- enlaces de hidrógeno;
- interacciones electrostáticas;
- interacciones de van der Waals;
- enlaces disulfuro en proteínas que contienen cisteínas apropiadamente posicionadas.

No existe una sola interacción responsable de toda la estructura.

La secuencia determina qué grupos químicos están disponibles, y esos grupos interactúan entre sí y con el ambiente.

## 8. Secuencia, estructura y mutaciones

Un cambio en un solo aminoácido puede modificar:

- el espacio disponible dentro de una proteína;
- las interacciones locales;
- la estabilidad;
- el reconocimiento molecular;
- y, en algunos casos, la función.

Compara estas dos cadenas laterales.

In [ ]:
#@title Comparar leucina y alanina

mols = [
    Chem.MolFromSmiles("CC(C)C[C@@H](N)C(=O)O"),
    Chem.MolFromSmiles("C[C@@H](N)C(=O)O")
]

legends = ["Leucina", "Alanina"]
img = Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(320, 240), legends=legends)
display(img)

print("Pregunta: si una leucina enterrada se reemplaza por alanina, qué podría cambiar?")
print("Pista: compara el tamaño de sus cadenas laterales.")

Una sustitución de una cadena lateral grande por una más pequeña puede crear **espacio adicional** dentro de la proteína.

En el workshop volveremos a esta idea con un sistema experimental real.

## 9. Proteínas, ligandos y reconocimiento molecular

Una proteína puede interactuar con otras moléculas en regiones específicas de su estructura.

**Ligando**  
Molécula que interactúa con una biomolécula.

**Binding site**  
Región donde ocurre la interacción.

**Binding pocket**  
Cavidad o región geométrica que puede acomodar un ligando.

La complementariedad de forma ayuda, pero no es suficiente.

También importan:

- propiedades químicas;
- interacciones intermoleculares;
- flexibilidad;
- solvente;
- dinámica molecular.

## 10. Las estructuras moleculares también son datos

Una imagen de una proteína no es el dato original.

Un archivo estructural contiene coordenadas tridimensionales:

| Átomo | x | y | z |
|---|---:|---:|---:|
| N | ... | ... | ... |
| CA | ... | ... | ... |
| C | ... | ... | ... |

El software utiliza esas coordenadas para generar representaciones como `Cartoon`, `Surface`, `Sticks` o `Spheres`.

De manera similar, las moléculas pequeñas pueden representarse usando texto.

## 11. SMILES: moléculas como texto

**SMILES** es una notación que representa la conectividad química de una molécula mediante texto.

No necesitas aprender a escribir SMILES durante este workshop. Solo necesitas reconocer que puede utilizarse como entrada para programas computacionales.

In [ ]:
#@title Seleccionar una molécula

molecule = "Ibuprofen" #@param ["Caffeine", "Acetaminophen", "Ibuprofen"]

molecules = {
    "Caffeine": "Cn1c(=O)c2c(ncn2C)n(C)c1=O",
    "Acetaminophen": "CC(=O)NC1=CC=C(O)C=C1",
    "Ibuprofen": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
}

smiles = molecules[molecule]
mol = Chem.MolFromSmiles(smiles)

display(Draw.MolToImage(mol, size=(460, 280)))
print(f"Molécula: {molecule}")
print(f"SMILES: {smiles}")

## 12. De una estructura a una propiedad

Los programas de quimioinformática pueden calcular descriptores a partir de una estructura molecular.

Antes de ejecutar la siguiente celda:

**Cuál de las tres moléculas crees que será más hidrofóbica?**

In [ ]:
#@title Comparar propiedades moleculares

rows = []
for name, smi in molecules.items():
    mol = Chem.MolFromSmiles(smi)
    rows.append({
        "Molécula": name,
        "MW (Da)": round(Descriptors.MolWt(mol), 2),
        "MolLogP": round(Crippen.MolLogP(mol), 2),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol)
    })

df = pd.DataFrame(rows)
display(df)

most_hydrophobic = df.loc[df["MolLogP"].idxmax(), "Molécula"]
print(f"Mayor MolLogP en este conjunto: {most_hydrophobic}")

Acabas de seguir el mismo patrón que utilizaremos durante todo el workshop:

**predecir → calcular → observar → interpretar**

El resultado computacional no reemplaza tu razonamiento. Te proporciona nueva evidencia para revisarlo.

## 13. El orden de las celdas importa

Cuando ejecutas una celda, sus variables y archivos permanecen temporalmente en la memoria del runtime.

Por eso, una celda posterior puede depender de algo creado anteriormente.

Si el runtime se reinicia o desconecta, esa memoria se pierde.

En ese caso:

**Runtime → Run all**

o vuelve a ejecutar el notebook desde el inicio.

## 14. Mini-repaso

Antes del workshop deberías poder reconocer estas ideas:

- Las proteínas están formadas por aminoácidos.
- Las cadenas laterales tienen propiedades químicas distintas.
- La secuencia corresponde a la estructura primaria.
- Hélices alfa y hojas beta son ejemplos de estructura secundaria.
- La estructura terciaria describe el fold tridimensional completo.
- Las proteínas pueden contener cavidades y sitios de unión.
- Un ligando es una molécula que interactúa con una biomolécula.
- La unión depende de forma, química, dinámica y ambiente.
- Un PDB contiene coordenadas moleculares.
- SMILES representa moléculas pequeñas como texto.
- Un notebook combina explicación, código y resultados.

## 15. Verificación final

Ejecuta la última celda para confirmar que tu entorno está listo.

In [ ]:
#@title Verificar que estás listo para el workshop

import os
import platform
import multiprocessing

checks = {
    "Google Colab": "google.colab" in sys.modules,
    "CPU runtime": multiprocessing.cpu_count() >= 1,
    "RDKit": importlib.util.find_spec("rdkit") is not None,
    "py3Dmol": importlib.util.find_spec("py3Dmol") is not None,
    "pandas": importlib.util.find_spec("pandas") is not None,
    "Estructura 1UBQ": os.path.exists("1UBQ.pdb")
}

width = max(len(k) for k in checks)
for name, ok in checks.items():
    status = "OK" if ok else "REVISAR"
    print(f"{name:<{width}}   {status}")

print()
print(f"CPUs disponibles: {multiprocessing.cpu_count()}")
print(f"Python: {platform.python_version()}")
print()
print("Preparación completada.")
print("No necesitas instalar nada localmente para comenzar el workshop.")

---

## Atribución y recursos

Este material fue diseñado para **InspiraSTEM 2026 | Bioquímica Computacional Aplicada**.

La estructura del flujo computacional y varias decisiones de diseño para Google Colab están inspiradas en el enfoque reproducible de **Cloud-Bind**, una plataforma de workflows de modelado molecular basados en la nube.

Herramientas utilizadas en esta preparación:

- RDKit — quimioinformática y representación molecular
- py3Dmol — visualización molecular interactiva
- RCSB Protein Data Bank — estructuras experimentales

Durante el workshop se introducirán y citarán las herramientas especializadas correspondientes en los notebooks donde se utilicen.